In [1]:
import torch
from torch import nn

torch.set_num_threads(2)
torch.manual_seed(7)

rng = torch.Generator().manual_seed(7)
words = "<start> a red green blue square circle <end>".split()
ids = torch.arange(24)
images, captions = [], []

for i in range(1200):
    color, shape, cx, cy, size = [
        int(torch.randint(low, high, (), generator=rng))
        for low, high in ((0, 3), (0, 2), (8, 17), (8, 17), (4, 8))
    ]
    dx, dy = ids[None, :] - cx, ids[:, None] - cy
    mask = (dx.abs() <= size) & (dy.abs() <= size) if shape == 0 else dx**2 + dy**2 <= size**2
    image = torch.rand(3, 24, 24, generator=rng) * 0.12
    image[color, mask] = 1
    images.append(image)
    captions.append([0, 1, color + 2, shape + 5, 7])

images, captions = torch.stack(images), torch.tensor(captions)
encoder = nn.Sequential(
    nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(3),
    nn.Flatten(), nn.Linear(288, 48), nn.Tanh()
)
embedding = nn.Embedding(len(words), 16)
decoder = nn.GRU(64, 48, batch_first=True)
head = nn.Linear(48, len(words))
model = nn.ModuleList([encoder, embedding, decoder, head])
optimizer = torch.optim.Adam(model.parameters(), lr=0.003)

for epoch in range(35):
    for batch in torch.randperm(1000).split(64):
        target = captions[batch]
        features = encoder(images[batch]).unsqueeze(1).expand(-1, target.size(1) - 1, -1)
        states, _ = decoder(torch.cat((embedding(target[:, :-1]), features), dim=-1))
        loss = nn.functional.cross_entropy(head(states).transpose(1, 2), target[:, 1:])
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

with torch.no_grad():
    features = encoder(images[1000:]).unsqueeze(1)
    hidden = None
    token = torch.zeros(len(images) - 1000, 1, dtype=torch.long)
    decoded = []
    for step in range(4):
        states, hidden = decoder(torch.cat((embedding(token), features), dim=-1), hidden)
        token = head(states).argmax(-1)
        decoded.append(token)
    predictions = torch.cat(decoded, dim=1)

print(f"Synthetic image test caption accuracy: {(predictions == captions[1000:, 1:]).all(1).float().mean():.1%}")
for expected, generated in zip(captions[1000:1006, 1:], predictions[:6]):
    actual = " ".join(words[i] for i in expected[:-1])
    predicted = " ".join(words[i] for i in generated if i not in (0, 7))
    print(f"Actual: {actual:16} Generated: {predicted}")

Synthetic image test caption accuracy: 100.0%
Actual: a red square     Generated: a red square
Actual: a green square   Generated: a green square
Actual: a blue square    Generated: a blue square
Actual: a red square     Generated: a red square
Actual: a blue circle    Generated: a blue circle
Actual: a blue square    Generated: a blue square
